In [ ]:
import pandas as pd

In [11]:
df = pd.read_csv("patient_records_50k.csv")

In [12]:
df.head() #200

,patient_id,name,age,gender,medical_conditions,current_medications,visit_id,visit_date,problem_description,doctor_notes,tests_ordered,test_results
0,P00000,Damon Moore,38,Male,Hypertension,Vitamin D; Lisinopril; Albuterol; Levothyroxine,V0001,2024-08-13,Muscle Cramps,Recommend tests: Lisinopril. | Encourage a Lev...,Blood Pressure Monitoring,Blood Pressure Monitoring: 5.1
1,P00000,Damon Moore,38,Male,Hypertension,Vitamin D; Lisinopril; Albuterol; Levothyroxine,V0002,2024-01-04,Fatigue,Recommend tests: Albuterol. | Increase dosage ...,Blood Pressure Monitoring; Lipid Profile; Elec...,Blood Pressure Monitoring: 8.78; Lipid Profile...
2,P00000,Damon Moore,38,Male,Hypertension,Vitamin D; Lisinopril; Albuterol; Levothyroxine,V0003,2024-05-25,Joint Pain,Schedule a follow-up in Atorvastatin weeks for...,Electrolyte Panel,Electrolyte Panel: 6.93
3,P00000,Damon Moore,38,Male,Hypertension,Vitamin D; Lisinopril; Albuterol; Levothyroxine,V0004,2024-07-01,Fatigue,Recommend tests: Albuterol. | Encourage a Lisi...,Nerve Conduction Test; Lipid Profile; Electrol...,Nerve Conduction Test: 1.21; Lipid Profile: 8....
4,P00000,Damon Moore,38,Male,Hypertension,Vitamin D; Lisinopril; Albuterol; Levothyroxine,V0005,2024-09-26,Numbness in Feet,Patient shows signs of Atorvastatin. Recommend...,Lipid Profile; Electrolyte Panel,Lipid Profile: 8.52; Electrolyte Panel: 5.5


In [13]:
(df.isnull().sum()/df.shape[0])*100

,0
patient_id,0.0
name,0.0
age,0.0
gender,0.0
medical_conditions,0.0
current_medications,0.0
visit_id,0.0
visit_date,0.0
problem_description,0.0
doctor_notes,0.0


In [ ]:
df.shape

In [5]:
df["patient_id"].value_counts()

,count
patient_id,
P49996,6
P00000,6
P00001,6
P49994,6
P00003,6
...,...
P49960,5
P49961,5
P49963,5


In [ ]:
df.shape

In [7]:
df = df.iloc[:100, :]
df.shape

(100, 12)

In [8]:
df.columns

Index(['patient_id', 'name', 'age', 'gender', 'medical_conditions',
       'current_medications', 'visit_id', 'visit_date', 'problem_description',
       'doctor_notes', 'tests_ordered', 'test_results'],
      dtype='object')

In [ ]:
df["combined"] = df["medical_conditions"]+ " " +df["current_medications"]+ " " +df["problem_description"]+ " " +df['doctor_notes']+ " " +df['tests_ordered']

In [ ]:
df.head(3)

In [ ]:
df["combined"][0]

# Vectorization (embedding)

In [ ]:
from sentence_transformers import SentenceTransformer

In [ ]:
model_name = "sentence-transformers/all-MiniLM-L6-v2"
model = SentenceTransformer(model_name)

100


In [ ]:
embedding = model.encode(df["combined"])

In [ ]:
embedding.shape

In [ ]:
embedding[0]

# Vector Store

In [ ]:
import torch
torch.cuda.is_available()

In [ ]:
import faiss

In [ ]:
vector_dimension = embedding[0].shape[0]

In [ ]:
quantizer = faiss.IndexFlatL2(vector_dimension)
index_ivf = faiss.IndexIVFFlat(quantizer, vector_dimension, 5)
index_ivf.train(embedding)
index_ivf.add(embedding)

In [ ]:
faiss.write_index(index_ivf, "index.faiss")

In [ ]:
index = faiss.read_index("index.faiss")

In [ ]:
def similar_records(query, k=5):
  query_embedd = model.encode([query])
  distance, indices = index.search(query_embedd, k=5)
  result = df.iloc[indices[0]]
  return result

In [ ]:
query = "what are test to be recommended for blood pressure"
query_embedd = model.encode([query])

In [ ]:
distance, indices = index.search(query_embedd, k=5)

In [ ]:
distance

In [ ]:
indices[0]

In [ ]:
df["doctor_notes"].iloc[indices[0]].tolist()

In [ ]:
endpoint = "https://unext8.openai.azure.com/"
model_name = "gpt-4.1-mini"
api_version = "2024-12-01-preview"
AZURE_OPENAI_API_KEY = "YOUR_AZURE_OPENAI_KEY"

In [ ]:
#!pip install langchain_openai

In [ ]:
from langchain_openai import AzureChatOpenAI
from langchain_core.messages import HumanMessage

In [ ]:
llm = AzureChatOpenAI(api_key=AZURE_OPENAI_API_KEY,
                azure_endpoint=endpoint,
                deployment_name=model_name,
                api_version=api_version)

In [ ]:
def generate_answe(query):
  retrieved_data = similar_records(query)
  context = "\n".join(retrieved_data['combined'].tolist())
  template = f"""use the following context to answer the question at the end.
  if you dont know the answer just say dont know,
  {context}

  Question: {query}
  answer:"""
  prompt = template.format(context=context, question=query)
  messages = [HumanMessage(content=prompt)]
  response = llm(messages)
  return response.content


In [ ]:
q = "what are the test are recommended for numbness"

In [ ]:
print(generate_answe(q))